In [5]:
import faiss
import pickle
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import pandas as pd
from tqdm import tqdm
import time

path_bge = "E:/my_project/local_models/bge-m3"
path_rerank = "E:/my_project/local_models/bge-reranker-base"

print("正在启动检索系统...")
device = "cuda" if torch.cuda.is_available() else "cpu"

if 'model' in locals():
    print(f"✅ 变量 'model' 已定义。类型: {type(model)}")
else:
    model = SentenceTransformer(path_bge, device=device)
    print("✅ 模型 BGE-M3 加载完成")


# 2. 加载 FAISS 索引 (你的数学坐标库)
# 这一步极快，毫秒级
if 'index' in locals():
    print(f"✅ 索引 'index' 已定义。类型: {type(index)}")
else:
    index = faiss.read_index("qasper_faiss.index")
    print(f"✅ 索引加载完成，包含 {index.ntotal} 条向量")
    
# 3. 加载文本数据 (你的内容库)
# 这一步也很快
if 'processed_docs' in locals():
    print(f"✅ 文本库 'processed_docs' 已加载。类型: {type(processed_docs)}")
else:
    with open("qasper_docs.pkl", "rb") as f:
        processed_docs = pickle.load(f) # <-- 必须缩进
    print(f"✅ 文本库加载完成")

# 4. 加载重排序模型 
if 'reranker' in locals():
    print(f"✅ 变量 'reranker' 已定义。类型: {type(reranker)}")
else:
    reranker = CrossEncoder(path_rerank, device=device)
    print("✅ 模型 bge-reranker-base 加载完成")
    
# 5. 初始化 BM25 (关键词检索)
# 注意：英文直接 .split()，如果是中文需要分词
if 'bm25' in locals():
    print(f"✅ 变量 'bm25' 已定义。类型: {type(bm25)}")
else:
    corpus_texts = [doc['text_content'] for doc in processed_docs]
    tokenized_corpus = [text.split(" ") for text in corpus_texts] 
    bm25 = BM25Okapi(tokenized_corpus)
    print("✅ BM25 索引构建完成")

print("\n🎉 系统准备就绪！")

正在启动检索系统...
✅ 变量 'model' 已定义。类型: <class 'sentence_transformers.SentenceTransformer.SentenceTransformer'>
✅ 索引 'index' 已定义。类型: <class 'faiss.swigfaiss_avx2.IndexFlatIP'>
✅ 文本库 'processed_docs' 已加载。类型: <class 'list'>
✅ 变量 'reranker' 已定义。类型: <class 'sentence_transformers.cross_encoder.CrossEncoder.CrossEncoder'>
✅ 变量 'bm25' 已定义。类型: <class 'rank_bm25.BM25Okapi'>

🎉 系统准备就绪！


In [2]:
import pandas as pd

# 1. 加载训练数据 
train_df = pd.read_csv('./dataset/train.csv')
print(f"加载训练集完成，共 {len(train_df)} 条测试题。")

# 2. 定义一个核心检索函数
# 输入：一个查询句子 (Claim)
# 输出：系统认为最相关的段落信息
def search_engine(query, top_k_retrieve=50, top_k_return=10):
    """
    流程：
    1. Vector 召回 Top-k
    2. BM25 召回 Top-k
    3. 取并集 (去重)
    4. Reranker 精排所有候选
    5. 返回 Top-k
    """
    
    # --- 路径 1: 向量检索 (Vector) ---
    q_vec = model.encode([query], normalize_embeddings=True)
    D, I = index.search(q_vec, top_k_retrieve)
    vector_candidates_idx = I[0]
    
    # --- 路径 2: 关键词检索 (BM25) ---
    tokenized_query = query.split(" ")
    # get_top_n 返回的是文档内容，我们需要索引，所以用 get_scores
    bm25_scores = bm25.get_scores(tokenized_query)
    # 获取分数最高的 top_k 个索引
    bm25_candidates_idx = np.argsort(bm25_scores)[::-1][:top_k_retrieve]
    
    # --- 合并候选集 (去重) ---
    # 使用 set 来合并两个来源的索引
    all_candidate_indices = list(set(vector_candidates_idx) | set(bm25_candidates_idx))
    
    # 过滤掉越界的索引 (以防万一)
    all_candidate_indices = [idx for idx in all_candidate_indices if idx < len(processed_docs)]
    
    if not all_candidate_indices:
        return []

    # --- 准备精排数据 ---
    rerank_pairs = []
    candidates_data = []
    
    for idx in all_candidate_indices:
        doc = processed_docs[idx]
        # 【技巧】精排时，把 Section Name 也拼进去让模型看，帮它区分章节
        # 格式： Claim: ... [SEP] Document: Section [Introduction] ...
        doc_text_with_meta = f"Section: {doc['section_name']} | Content: {doc['text_content']}"
        
        rerank_pairs.append([query, doc_text_with_meta])
        candidates_data.append(doc)
        
    # --- 路径 3: Cross-Encoder 精排 ---
    scores = reranker.predict(rerank_pairs)
    
    # 排序
    results_with_scores = sorted(
        zip(candidates_data, scores), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    # 返回 Top K
    return [item[0] for item in results_with_scores[:top_k_return]]

# 测试一下
test_query = "Human perception can be altered by using subtle perturbations."
result = search_engine(test_query)
print("\n【单条测试】")
print(f"查询: {test_query}")
# Access the first element of the list using [0]
print(f"找到的段落: {result[0]['section_name']} | ID: {result[0]['paper_id']}")

# 假设 top_k_return=3
retrieved_contexts = []

# 遍历精排后的 Top-K 结果
for i, doc in enumerate(result):
    # 格式化每个文档，以便 LLM 区分
    context_text = f"【文档 {i+1} - 章节: {doc['section_name']}】\n内容: {doc['text_content']}\n"
    retrieved_contexts.append(context_text)

# 合并成一个大的字符串
final_context = "\n---\n".join(retrieved_contexts)

print("\n【合并后的上下文】")
print(final_context)

# 接着，将 final_context 和 original_query 一起构建 LLM 的 Prompt
# llm_prompt = f"根据以下提供的上下文信息回答问题：{query}\n\n上下文:\n{final_context}"
# llm.generate(llm_prompt)

加载训练集完成，共 1000 条测试题。

【单条测试】
查询: Human perception can be altered by using subtle perturbations.
找到的段落: Conclusion | ID: 1906.0104

【合并后的上下文】
【文档 1 - 章节: Conclusion】
内容: Content: Our work suggests that for a significant fraction of natural speech, human perception can be altered by using subtle, learnable perturbations. This is an initial step towards exploring the density of illusionable phenomenon for humans, and examining the extent to which human perception may be vulnerable to security risks like those that adversarial examples present for ML systems. We hope our work inspires future investigations into the discovery, generation, and quantification of multimodal and unimodal audiovisual and auditory illusions for humans. There exist many open research questions on when and why humans are susceptible to various types of illusions, how to model the illusionability of natural language, and how natural language can be made more robust to illusory perturbations. Additionally, we hope su

In [3]:
import json
from openai import OpenAI

# 接入模型(由于个人隐私原因不列出相关API URL)
API_KEY = 
BASE_URL = 
MODEL_NAME = "deepseek-ai/DeepSeek-V3"

try:
    # 实例化 client
    client = OpenAI(
        base_url=BASE_URL,
        api_key=API_KEY
    )
except Exception as e:
    # 如果导入或初始化失败，输出错误并退出
    print(f"客户端初始化失败: {e}")
    # 可以在这里添加 raise 或 sys.exit() 停止后续执行

# 初始化客户端
def llm_verify(claim, evidence_text):
    """
    输入:
        claim: 待验证的陈述
        evidence_text: 检索到的文献段落
    输出:
        pred_label (0/1), reasoning (str), best_evidence_id (str, e.g., "证据 3")
    """
    
    # 1. 构建 Prompt (提示词)
    system_prompt = """你是一个严谨的学术审稿人。请基于【证据】验证【陈述】。

【核心原则：宁缺毋滥】
只有当证据**明确支持**且**无后续反转或关键限制条件被忽略**时，才标记为 1。

【多证据处理】
证据段落可能来自多个来源（编号为 "证据 1", "证据 2" 等）。请务必**综合评估所有证据**。

---
【判断标准】

1. Label 1 (True): 陈述是证据的同义改写、归纳或逻辑推论，且没有遗漏任何关键限制。
2. Label 0 (False): 存在以下任一情况：
    - 矛盾/反驳：证据中存在转折词 ("However", "But") 否定了陈述。
    - 引用非事实：证据说 "Previous studies suggest X..."，但陈述声称 "X is true"。引用不等于事实。
    - 过度泛化：证据中存在限制条件，但陈述将其泛化。
    - 幻觉/未提及。

---
【强制思维链 (Reasoning) 要求】

在分析时，请必须回答以下问题，并将分析结果体现在你的 reasoning 中：
1. 证据中的信息是作者的观点，还是作者引用的他人观点？
2. 证据周围有没有转折词 (But/However) 否定了陈述？
3. 陈述是否漏掉了证据中的关键限制条件？

请务必输出包含 "reasoning" (分析原因), "label" (判别结果), 以及 **"best_evidence_id"** (支持你判断的【证据 X】编号，例如 "证据 1"、"证据 3" 等) 的 JSON。如果无法判断或证据不足，请将 "best_evidence_id" 设为 "N/A"。

【示例 1】
例如：以下是review_id为1906.01040⽂献在Conclusion中的⼀个部分：
证据: "Our work suggests that for a significant fraction of natural speech, human perception can be altered by using subtle, learnable perturbations. 
    This is an initial step towards exploring the density of illusionable phenomenon for humans, and examining the extent to which human
    perception may be vulnerable to security risks like those that adversarial examples present for Msystems."
陈述: "Human perception can be altered by using subtle, learnable perturbations for a significant fraction of natural speech."
输出: {"reasoning": "证那么这个陈述根据⽂本⼤意，容易判断为符合的陈述，因为它是⽂章的第⼀句的转述。", "label": 1}

【示例 2】
以下是review_id 为1912.13337 的⽂献在 Dataset Probes and Construction ::: WordNetQA::: Perturbations and Semantic Clusters 部分的⽂本
证据: "Based on how we generate data, for each concept (i.e., atomic WordNet synset) and probe type(i.e., definitions, hypernymy, etc.),
    we have a wide variety of questions related to that manipulate1) the complexity of reasoning that is involved (e.g., the number of inferentiahops) and;
    2) thetypes of distractors (or distractor perturbations) that are employed. We call such sets semanticclusters. 
    As we describe in the next section, semantic clusters allow us to devise new types of evaluation that reveal 
    whether models have comprehensive and consistent knowledge of targetconcepts 
    (e.g., evaluating whether a model can correctly answer several questions associated witha concept, as opposed to a few disjoint instances)."
陈述: "Semantic clusters are used to evaluate whether models can correctly answer a few disjoint instances of questions associated with a concept, 
    rather than assessing comprehensive and consistent knowledge of the target concept."
输出: {"reasoning": "在⽂本中，semantic clusters 的⽬的是为了评估模型是否具有⽬标概念的全⾯和⼀致的知识，⽽不仅仅是评估模型是否能够回答⼏个分离的实例问题。
    ⽂本明确指出，semantic clusters ⽤于设计新的评估类型，旨在揭⽰模型是否对⽬标概念有全⾯⼀致的知识，⽽不是仅仅回答⼀些不相关的实例问题。
    所以这种以偏盖全的陈述，会被判定为错误的陈述，是不符合原⽂陈述。", "label": 0}

请务必输出包含 "reasoning" (分析原因), "label" (判别结果), 以及 "best_evidence_id" (支持你判断的【证据 X】编号，例如：**只需要输出数字**，如 "1" 或 "3") 的 JSON。如果无法判断或证据不足，请将 "best_evidence_id" 设为 "N/A"。
现在请对以下输入进行判别，仅输出 JSON。"""

    user_prompt = f"【文献证据】：\n{evidence_text}\n\n【陈述】：\n{claim}\n\n请输出 JSON 结果："

    try:
        # 'client' 现在在全局作用域已定义，函数可以访问它
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"} 
        )
        content = response.choices[0].message.content
        result_json = json.loads(content)
        return (
            result_json.get('label', 0), 
            result_json.get('reasoning', "无详细分析"),
            result_json.get('best_evidence_id', "N/A")
        )
    except Exception as e:
        # 如果 API 调用失败，会捕获这里的错误
        return 0, f"API 调用 Error: {str(e)}"


In [4]:
# ====================== 配置和加载 ======================
# 1. 加载测试集 (假设文件名为 test.csv)
try:
    test_df = pd.read_csv('./dataset/test.csv')
    print(f"✅ 加载测试集完成，共 {len(test_df)} 条数据。")
except FileNotFoundError:
    print("❌ 错误：未找到 ./dataset/test.csv 文件，请检查路径。")
    # 如果找不到测试集，可以使用训练集的子集进行演示
    if 'train_df' in locals():
        test_df = train_df.sample(n=10, random_state=1)
        print("    --> 切换为训练集10个样本的子集进行演示。")
    else:
        # 如果 train_df 也没加载，则退出
        exit() 

# 2. 准备结果容器
submission_data = []

✅ 加载测试集完成，共 1984 条数据。


In [6]:
import pandas as pd
from tqdm import tqdm
import time

print("\n🏁 开始对测试集进行 RAG 全链路预测...")
print("----------------------------------------------------------")

submission_data = [] # 假设 submission_data 在循环前已定义为列表
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    data_id = row['ID']
    claim = row['raw_input']
    
    # 默认值，以防检索失败或 LLM 证据选择失败
    review_id = ""
    section_name = ""
    pred_label = 0 
    
    try:
        # --- Step 1: 混合检索 (获取 Top 5 结果) ---
        results_list = search_engine(claim)
        
        if not results_list:
            print(f"⚠️ ID {data_id}: 未检索到文档。")
            evidence_text = "No relevant document found."
            # 此时 review_id 和 section_name 保持默认 ""
        else:
            # 1.1 构建增强上下文 (Top 5 拼接)
            top_k = 10
            valid_docs = results_list[:top_k]
            formatted_contexts = []
            
            for i, doc in enumerate(valid_docs):
                # 在每个文档前加上清晰的标签和元数据，编号从 1 开始
                context_part = (
                    f"【证据 {i+1}】来源: {doc.get('paper_id', 'N/A')} | 章节: {doc.get('section_name', 'N/A')}\n"
                    f"内容: {doc['text_content']}"
                )
                formatted_contexts.append(context_part)
                
            evidence_text = "\n\n=====\n\n".join(formatted_contexts)
            
            # 默认使用 Reranker 的 Top-1 结果作为回退
            top1_doc = valid_docs[0]
            review_id = str(top1_doc['paper_id'])
            section_name = str(top1_doc['section_name'])

        # --- Step 2: 验证 (LLM) ---
        # LLM 现在返回 pred_label, reasoning, 和 best_evidence_id
        pred_label, reasoning, best_evidence_id = llm_verify(claim, evidence_text)
        
        # --- Step 2.1: 动态选择提交的证据来源 (优化后) ---
        if best_evidence_id and best_evidence_id.startswith("证据"):
            try:
                # 优化点 1: 使用正则表达式提取第一个数字
                match = re.search(r'\d+', best_evidence_id)
                
                if match:
                    best_doc_index = int(match.group(0)) - 1 # 提取匹配到的数字，并转为 0-based 索引
        
                    # 优化点 2: 保持边界检查
                    if 0 <= best_doc_index < len(valid_docs):
                        selected_doc = valid_docs[best_doc_index]
                        review_id = str(selected_doc['paper_id'])
                        section_name = str(selected_doc['section_name'])
                    else:
                         print(f"⚠️ ID {data_id}: LLM 选出的证据编号 {best_doc_index+1} 超出 Top-K 范围。回退到 Top-1。")
                else:
                    raise ValueError("未在 best_evidence_id 中找到数字编号。")
        
            except Exception as parse_e:
                # 此时的 parse_e 可能是 ValueError (未找到数字) 或其他错误
                print(f"⚠️ ID {data_id}: 解析 best_evidence_id 失败 ({best_evidence_id})。回退到 Top-1。")
        
        # --- Step 3: 收集结果 ---
        submission_data.append({
            "ID": data_id,
            "review_id": review_id,
            "section_name": section_name,
            "label": pred_label
        })
        
        time.sleep(0.5)

    except Exception as e:
        print(f"❌ Error at ID {data_id}: {e}，该条数据使用默认值。")
        # 发生错误时，使用当前提取到的/默认的 ID 和 Section 名称
        submission_data.append({
            "ID": data_id,
            "review_id": review_id,
            "section_name": section_name,
            "label": 0 
        })
        time.sleep(0.5)

# ====================== 生成提交文件 ======================
submission_df = pd.DataFrame(submission_data)
submission_df = submission_df[["ID", "review_id", "section_name", "label"]]
output_filename = "submission.csv"
submission_df.to_csv(output_filename, index=False)

print("\n" + "="*50)
print(f"预测完成")
print(f"总预测条数: {len(submission_df)}")
print(f"文件已保存为: {output_filename}")
print("==================================================")


🏁 开始对测试集进行 RAG 全链路预测...
----------------------------------------------------------


 35%|██████████████████████████▎                                               | 704/1984 [5:11:13<12:29:55, 35.15s/it]

❌ Error at ID 704: not enough values to unpack (expected 3, got 2)，该条数据使用默认值。


 40%|█████████████████████████████▉                                            | 803/1984 [5:55:06<11:35:07, 35.32s/it]

❌ Error at ID 803: not enough values to unpack (expected 3, got 2)，该条数据使用默认值。


 41%|█████████████████████████████▉                                            | 804/1984 [5:55:40<11:28:11, 34.99s/it]

❌ Error at ID 804: not enough values to unpack (expected 3, got 2)，该条数据使用默认值。


 46%|██████████████████████████████████▌                                        | 914/1984 [6:49:15<7:59:06, 26.87s/it]


KeyboardInterrupt: 

In [12]:
submission_df.head()

,ID,review_id,section_name,label
0,0,2003.08385,Related Work ::: Multi-Target Stance Detection,0
1,1,1604.06076,Results,0
2,2,1910.11768,Related Work,1
3,3,1908.09355,Conclusion,0
4,4,1901.03438,Analysis Set,1


# 训练调试

In [34]:
import pandas as pd
from tqdm import tqdm
import time

# ================= 配置区域 =================
SAMPLE_SIZE = 50 
# ===========================================

if 'train_df' not in locals():
    train_df = pd.read_csv('./dataset/train.csv')

test_samples = train_df.sample(n=SAMPLE_SIZE, random_state=42).copy()

print(f"🏁 开始 RAG 全链路严格评估 (Top 3 拼接版)...")
print("--------------------------------------------------")

results = []
strict_correct_count = 0 # 严格匹配计数
retrieval_correct_count = 0 # 仅检索正确计数

for idx, row in tqdm(test_samples.iterrows(), total=SAMPLE_SIZE):
    claim = row['ans']        
    true_label = row['label'] 
    true_id = str(row['review_id'])
    true_section = str(row['section_name']) # 确保格式一致
    
    try:
        # --- Step 1: 检索 ---
        results_list = search_engine(claim)
        
        if not results_list:
            print(f"⚠️ Index {idx}: 未检索到文档")
            continue
        
        # 1.1 检索命中判断 (以 Top 1 结果为准进行检索评估)
        top1_doc = results_list[0]
        pred_id = str(top1_doc['paper_id'])
        pred_section = str(top1_doc['section_name'])
        
        # 判断检索是否正确 (ID 和 章节名都必须对)
        is_retrieval_correct = (pred_id == true_id and pred_section == true_section)
        if is_retrieval_correct:
            retrieval_correct_count += 1

        # 1.2 构建增强上下文 (Top 3 拼接)
        # 即使 Top 1 没找对，拼接 Top 3 也有可能把正确段落包含进来，所以这里继续做
        top_k = 3 
        valid_docs = results_list[:top_k]
        evidence_text = "\n---\n".join([doc['text_content'] for doc in valid_docs])
        
        # --- Step 2: 验证 (LLM) ---
        # 使用拼接后的长文本进行推理
        pred_label, reasoning = llm_verify(claim, evidence_text)
        
        # --- Step 3: 严格判分 ---
        # 逻辑：检索必须对 AND 推理必须对
        is_reasoning_correct = (pred_label == true_label)
        is_strictly_correct = (is_retrieval_correct and is_reasoning_correct)
        
        if is_strictly_correct:
            strict_correct_count += 1
            
        results.append({
            "claim": claim[:30] + "...",
            "retrieved_id": pred_id,
            "true_id": true_id,
            "retrieved_section": pred_section,
            "true_section": true_section,
            "is_retrieval_correct": is_retrieval_correct, # 检索对了没
            "true_label": true_label,
            "pred_label": pred_label,
            "is_reasoning_correct": is_reasoning_correct, # 推理对了没
            "is_strictly_correct": is_strictly_correct,   # 两个都对才算对
            "reasoning": reasoning
        })
        
        time.sleep(0.5)

    except Exception as e:
        print(f"❌ Error at index {idx}: {e}")

# 3. 输出详细战报
print("\n" + "="*50)
print(f"📊 【严格模式战报】")
print(f"样本数量: {SAMPLE_SIZE}")
print(f"1. 检索准确率 (Recall@1): {retrieval_correct_count / SAMPLE_SIZE:.2%}")
print(f"2. 端到端严格准确率 (Strict Accuracy): {strict_correct_count / SAMPLE_SIZE:.2%}")
print("   (要求: 文献ID对 + 章节名对 + 最终判决对)")
print("="*50)

# 4. 更有针对性的错题分析
print("\n🔍 【丢分原因分析 (前 3 例)】")
wrong_cases = [r for r in results if not r['is_strictly_correct']]

for i, case in enumerate(wrong_cases[:3]):
    print(f"错题 {i+1}:")
    if not case['is_retrieval_correct']:
        print(f"❌ 检索错误 (Retrieval Error)")
        print(f"   预测位置: {case['retrieved_id']} | {case['retrieved_section']}")
        print(f"   真实位置: {case['true_id']} | {case['true_section']}")
    elif not case['is_reasoning_correct']:
        print(f"❌ 推理错误 (Reasoning Error) - 检索已命中，但大模型判错了")
        print(f"   大模型分析: {case['reasoning']}")
        print(f"   预测 Label: {case['pred_label']} | 真实 Label: {case['true_label']}")
    print("-" * 40)

🏁 开始 RAG 全链路严格评估 (Top 3 拼接版)...
--------------------------------------------------


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [34:25<00:00, 41.31s/it]


📊 【严格模式战报】
样本数量: 50
1. 检索准确率 (Recall@1): 90.00%
2. 端到端严格准确率 (Strict Accuracy): 88.00%
   (要求: 文献ID对 + 章节名对 + 最终判决对)

🔍 【丢分原因分析 (前 3 例)】
错题 1:
❌ 推理错误 (Reasoning Error) - 检索已命中，但大模型判错了
   大模型分析: 证据中的第一段明确指出知识图谱通常被解释为由离散的三元组（entityA, relationX, entityB）组成，这与陈述完全一致。此外，没有后续的反转或限制条件否定这一观点。
   预测 Label: 1 | 真实 Label: 0
----------------------------------------
错题 2:
❌ 检索错误 (Retrieval Error)
   预测位置: 1608.01972 | PubMed Experiments
   真实位置: 1608.01972 | Conclusion
----------------------------------------
错题 3:
❌ 检索错误 (Retrieval Error)
   预测位置: 2002.06854 | Experiments and Results ::: Evaluation Metrics and Baselines
   真实位置: 2002.06854 | Experiments and Results ::: Rating Prediction
----------------------------------------


In [ ]:
from tqdm import tqdm

correct_count = 0
total_count = 0
bad_cases = [] # 记录错题本

print("\n开始全量评估 (这可能需要一小会儿)...")

for i, row in tqdm(train_df.iterrows(), total=len(train_df)):
    # 1.以此题干作为查询
    claim = row['ans'] # 注意：根据你的描述，ans列是陈述
    
    # 2. 获取标准答案
    true_id = str(row['review_id']) # 转字符串防格式不一
    true_section = row['section_name']
    
    # 3. 系统作答
    pred_result = search_engine(claim)
    pred_result = pred_result[0]
    pred_id = str(pred_result['paper_id'])
    pred_section = pred_result['section_name']
    
    # 4. 判分 (只有 ID 和 章节名 都对才算对)
    if pred_id == true_id and pred_section == true_section:
        correct_count += 1
    else:
        # 记录错题，方便后续分析
        bad_cases.append({
            "claim": claim,
            "true": f"{true_id} | {true_section}",
            "pred": f"{pred_id} | {pred_section}",
            "pred_text": pred_result['text_content'][:100] # 看看找错成啥了
        })
        
    total_count += 1

# 5. 输出成绩单
accuracy = correct_count / total_count
print("\n" + "="*40)
print(f"【考试成绩单】")
print(f"总题数: {total_count}")
print(f"答对题数: {correct_count}")
print(f"检索准确率 (Recall@1): {accuracy:.2%}")
print("="*40)

In [24]:
bad_cases

[{'claim': 'The dataset contains sentences and paragraphs from the Open American National Corpus (OANC) belonging to 5 genres: newswire, letters, journal, non-fiction, and fiction.',
  'true': '1809.08731 | Dataset',
  'pred': '1904.10503 | Datasets',
  'pred_text': 'Content: We evaluate our model on two publicly available datasets. The statistics for both are shown'},
 {'claim': 'The evaluation protocol involves ranking valid test triples and corrupted triples in ascending order of their scores.',
  'true': '1808.04122 | Experimental setup',
  'pred': '1712.02121 | Evaluation protocol',
  'pred_text': 'Section: Evaluation protocol\nContent: In the KB completion or link prediction task BIBREF13 , the pu'},
 {'claim': 'BIBREF15 introduced the task of detecting state changes in recipes using an entity-centric memory network neural architecture for simulating action dynamics.',
  'true': '1909.02635 | Background: Process Understanding',
  'pred': '1909.08859 | Procedural Reasoning Network